# TP 3 — Pipeline automatisé de gestion documentaire
## Clinique « La Paix » — Yopougon

Ce notebook présente et exécute, partie par partie, le pipeline demandé :

```
Réception → Tri → Nettoyage → Extraction → Analyse → Rapport → Archivage
```

Chaque étape est un script Python distinct (`01_recuperer.py` … `07_sauvegarder.py`), comme l'exige l'énoncé.
Le notebook les exécute avec `%run` et affiche les résultats intermédiaires.
Les paramètres sont centralisés dans `config.py` et les fonctions réutilisables dans `utils.py`.

| Fichier | Rôle |
|---|---|
| `main.py` | Orchestration (`subprocess.run(..., check=True)`) et planification (`schedule`) |
| `simuler_donnees.py` | Génération des 60 fichiers de test |
| `01_recuperer.py` | Préparation / récupération des données |
| `02_trier.py` | Tri des fichiers |
| `03_nettoyer.py` | Normalisation des noms |
| `04_extraire.py` | Extraction des données des logs |
| `05_analyser.py` | Calcul des statistiques |
| `06_exporter.py` | Génération des rapports |
| `07_sauvegarder.py` | Archivage et nettoyage |
| `config.py` / `utils.py` | Configuration centralisée / fonctions communes |

## Préparation

Installation de la dépendance (une seule fois) puis imports.
Si `REPARTIR_DE_ZERO = True`, le dossier `data/` (données générées) est supprimé pour rejouer le TP depuis le début.

In [ ]:
%pip install -q schedule

In [ ]:
import json
import shutil
import zipfile
from datetime import date
from pathlib import Path
from pprint import pprint

import config
import utils

REPARTIR_DE_ZERO = True

if REPARTIR_DE_ZERO and config.DATA_DIR.exists():
    shutil.rmtree(config.DATA_DIR)
    print("Dossier data/ supprimé : on repart de zéro.")

print("Dossier du projet :", config.BASE_DIR)

## Partie 8 — Configuration et fonctions communes

`config.py` regroupe les chemins, extensions, préfixes, motifs regex, l'heure d'exécution et la durée de conservation.

In [ ]:
print("HEURE_EXECUTION   :", config.HEURE_EXECUTION)
print("DUREE_ARCHIVAGE   :", config.DUREE_ARCHIVAGE, "jours")
print("Extensions        :", config.EXTENSIONS_CATEGORIES)
print("Préfixes          :", config.PREFIXES)
print("Regex des logs    :", config.REGEX_LOG)
print("Étapes du pipeline:", config.ETAPES)

`utils.py` contient notamment `creer_logger()`, `normaliser_nom()` et `extraire_logs()`.
Toutes les étapes écrivent dans le même journal : `logs/pipeline_YYYY-MM-DD.log`.

In [ ]:
logger = utils.creer_logger("notebook")
logger.info("Démarrage du notebook")
print("Journal :", config.LOGS_DIR / f"pipeline_{date.today():%Y-%m-%d}.log")

## Partie 1 — Simulation et préparation des données

`simuler_donnees.py` crée l'arborescence `data/` et génère **60 fichiers** (PDF, CSV, JPG, PNG, XLSX, LOG, TMP)
aux noms volontairement irréguliers, dont des journaux d'admission contenant quelques lignes mal formées.

In [ ]:
%run simuler_donnees.py

In [ ]:
fichiers_bruts = sorted(f.name for f in config.BRUT_DIR.iterdir() if f.is_file())
print(len(fichiers_bruts), "fichiers dans data/brut/ :\n")
for nom in fichiers_bruts:
    print("  ", nom)

`01_recuperer.py` vérifie l'arborescence et compte les fichiers reçus
(si `data/brut/` est vide, il lance lui-même la simulation).

In [ ]:
%run 01_recuperer.py

## Partie 2 — Tri automatique des fichiers

`02_trier.py` parcourt `data/brut/`, détermine la catégorie de chaque fichier à partir de son **préfixe** et de son
**extension**, puis le déplace avec `shutil.move()`. Les `.tmp` sont identifiés comme temporaires.

In [ ]:
for nom in ["Ordonnance Dr Koffi 01-09-2026.pdf", "IMG_20260901_123456.jpg",
            "Facture PATIENT 01-09-2026.xlsx", "scanner_temp_01.tmp"]:
    print(f"{nom:40} -> {utils.determiner_categorie(nom) or 'temporaire'}")

In [ ]:
%run 02_trier.py

In [ ]:
for categorie in config.CATEGORIES:
    n = len(list((config.BRUT_DIR / categorie).iterdir()))
    print(f"data/brut/{categorie:12} : {n} fichier(s)")
print("Temporaires restés dans data/brut/ :", len(list(config.BRUT_DIR.glob("*.tmp"))))

## Partie 3 — Normalisation des noms de fichiers

`normaliser_nom(nom)` : minuscules, espaces et tirets → `_`, dates → `YYYY-MM-DD`, suppression des caractères spéciaux,
gestion des accents, conservation de l'extension, préfixes harmonisés (`ordo_`, `analyse_`, `radio_`, `facture_`)
et suffixe `_2`, `_3`… pour éviter les doublons.

In [ ]:
exemples = [
    "Ordonnance Dr Koffi 01-09-2026.pdf",
    "ANALYSE SANG 01/09/2026.csv",
    "radio thorax 2026-09-01.jpg",
    "Facture PATIENT 01-09-2026.xlsx",
    "IMG_20260901_123456.jpg",
    "Radiographie CRÂNE 03 09 2026.png",
    "résultats labo glycémie 20260903.csv",
]
for nom in exemples:
    print(f"{nom:40} -> {utils.normaliser_nom(nom)}")

`03_nettoyer.py` renomme chaque fichier avec `Path.rename()` et le range dans `data/propre/<catégorie>/`.

In [ ]:
%run 03_nettoyer.py

In [ ]:
for categorie in config.CATEGORIES:
    noms = sorted(f.name for f in (config.PROPRE_DIR / categorie).iterdir())
    print(f"\n{categorie} ({len(noms)})")
    for nom in noms:
        print("  ", nom)

## Partie 4 — Extraction des informations des logs

`extraire_logs(chemin)` lit le fichier ligne par ligne avec une expression régulière à **groupes nommés**
(`date`, `heure`, `patient`, `ip`, `action`). Une ligne mal formée provoque un `logging.warning()` puis est ignorée.

In [ ]:
journal = sorted((config.PROPRE_DIR / "logs").glob("*.log"))[0]
evenements = utils.extraire_logs(journal, logger)
print(f"\n{len(evenements)} événements extraits de {journal.name}. Premier résultat :")
pprint(evenements[0])

In [ ]:
%run 04_extraire.py

In [ ]:
lignes = utils.lire_csv(config.LOGS_EXTRAITS_CSV)
print(len(lignes), "lignes dans data/propre/logs_extraits.csv\n")
for ligne in lignes[:8]:
    print(ligne)

## Partie 5 — Analyse statistique

`05_analyser.py` calcule les statistiques sur les documents (nombre, taille en Mo) et sur l'activité
(admissions, consultations, sorties, événements par jour, top 5 patients, répartition horaire),
puis enregistre le dictionnaire `statistiques` dans `data/propre/statistiques.json`.

In [ ]:
%run 05_analyser.py

In [ ]:
statistiques = json.loads(config.STATISTIQUES_JSON.read_text(encoding="utf-8"))
print("Clés :", list(statistiques))
pprint(statistiques, sort_dicts=False)

### Répartition horaire

Elle permet d'identifier les périodes de forte activité.

In [ ]:
repartition = statistiques["horaires"]["repartition"]
maxi = max(repartition.values())
for heure, n in repartition.items():
    print(f"{heure} | {'█' * round(n * 40 / maxi):<40} {n}")
print("\nPic d'activité :", statistiques["horaires"]["pic"])

## Partie 6 — Génération des rapports

`06_exporter.py` produit `data/rapports/rapport_YYYY-MM-DD.csv` et `data/rapports/rapport_YYYY-MM-DD.txt`
(lisible par un responsable non informaticien).

In [ ]:
%run 06_exporter.py

In [ ]:
rapport_txt = config.RAPPORTS_DIR / f"rapport_{date.today():%Y-%m-%d}.txt"
print(rapport_txt.read_text(encoding="utf-8"))

In [ ]:
rapport_csv = config.RAPPORTS_DIR / f"rapport_{date.today():%Y-%m-%d}.csv"
for ligne in utils.lire_csv(rapport_csv):
    print(f"{ligne['section']:22} | {ligne['indicateur']:38} | {ligne['valeur']:>6} {ligne['unite']}")

## Partie 7 — Sauvegarde et archivage

`07_sauvegarder.py` crée `data/archives/archive_YYYY-MM-DD.zip` à partir de `data/propre/`,
supprime les fichiers `.tmp` et ne conserve que les archives des 7 derniers jours.

In [ ]:
%run 07_sauvegarder.py

In [ ]:
for archive in sorted(config.ARCHIVES_DIR.glob("*.zip")):
    with zipfile.ZipFile(archive) as zf:
        print(f"{archive.name} : {len(zf.namelist())} fichiers, {archive.stat().st_size / 1024**2:.2f} Mo")
print("Fichiers .tmp restants :", len(list(config.DATA_DIR.rglob("*.tmp"))))

## Partie 9 — Orchestration et planification

`main.py` lance les 7 scripts dans l'ordre avec `subprocess.run(..., check=True)`.
Si une étape se termine avec un code différent de 0 : `logging.error()` puis arrêt propre.

On relance ici le pipeline complet d'un seul coup (les données sont régénérées car `data/brut/` est vide).

In [ ]:
from main import executer_pipeline

succes = executer_pipeline()
print("Pipeline réussi :", succes)

Les messages des étapes lancées par `subprocess` ne s'affichent pas dans le notebook, mais ils sont tous dans le journal :

In [ ]:
journal_pipeline = config.LOGS_DIR / f"pipeline_{date.today():%Y-%m-%d}.log"
lignes = journal_pipeline.read_text(encoding="utf-8").splitlines()
debut = max(i for i, l in enumerate(lignes) if "DÉBUT DU PIPELINE" in l)
print("\n".join(lignes[debut - 1:]))

### Planification quotidienne à 23h30

Dans `main.py`, la planification utilise la bibliothèque `schedule` :

```python
schedule.every().day.at("23:30").do(executer_pipeline)

while True:
    schedule.run_pending()
    time.sleep(1)
```

La boucle infinie bloquerait le notebook : en production, on lance dans un terminal :

```
python main.py               # attend 23:30 chaque jour
python main.py --maintenant  # exécution immédiate
```

La cellule suivante montre simplement que la tâche est bien programmée, sans lancer la boucle.

In [ ]:
import schedule

schedule.clear()
schedule.every().day.at(config.HEURE_EXECUTION).do(executer_pipeline)
for tache in schedule.get_jobs():
    print(tache)
    print("Prochaine exécution :", tache.next_run)
schedule.clear()

## Conclusion

Le pipeline traite automatiquement les documents reçus par la clinique :
tri, normalisation des noms, extraction des journaux d'admission, statistiques, rapports pour la direction
et archivage avec rotation sur 7 jours. Chaque opération est tracée dans `logs/pipeline_YYYY-MM-DD.log`.